# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters

In [ ]:
N = 10
MAX_RECORDINGS = 10000
CLIPS_PER_SPECIES = 2500
MIN_XC_RECORDINGS = 100
BIRDNET_THRESHOLD = 0.92

ACTIVE_COLLECTIONS = ["diff_genus"] # ["diff_family", "diff_genus", "diff_species"]

# Select species

In [ ]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS,dl_more=True)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

Species pool: 440 species with 100 XC recordings


In [ ]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
 genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order  min_top_n_rec
Turdus         13                 1                  1                 1            121

Potential families (need >=N distinct genera):
                              family  n_species  n_distinct_genus  n_distinct_family  n_distinct_order  min_top_n_rec
     Tyrannidae (Tyrant Flycatchers)         24                20                  1                 1            159
   Thamnophilidae (Typical Antbirds)         14                12                  1                 1            103
               Troglodytidae (Wrens)         21                11                  1                 1            113
  Passerellidae (New World Sparrows)         19                11                  1                 1            108
      Parulidae (New World Warblers)         24                10                  1                 1            106
Muscicapidae (Old World Flycatc

In [ ]:
TARGET_GENUS  = "Turdus"
TARGET_FAMILY = "Fringillidae (Finches, Euphonias, and Allies)"
TARGET_ORDER  = "Passeriformes"

AVOID_DIFF_SPECIES = []
AVOID_DIFF_GENUS = []
AVOID_DIFF_FAMILY = []

In [ ]:
collections: dict[str, list[str]] = {}
diff_species = select_same_genus(TARGET_GENUS, N, pool, avoid=AVOID_DIFF_SPECIES)
diff_genus = select_same_family(TARGET_FAMILY, N, pool, avoid=AVOID_DIFF_GENUS)
diff_family = select_same_order(TARGET_ORDER, N, pool, avoid=AVOID_DIFF_FAMILY)
if "diff_species" in ACTIVE_COLLECTIONS:
    collections["diff_species"] = [s.scientific_name for s in diff_species]
if "diff_genus" in ACTIVE_COLLECTIONS:
    collections["diff_genus"] = [s.scientific_name for s in diff_genus]
if "diff_family" in ACTIVE_COLLECTIONS:
    collections["diff_family"] = [s.scientific_name for s in diff_family]

collections_to_use = {i_collection: collections[i_collection] for i_collection in ACTIVE_COLLECTIONS}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for n in names:
        print(f"  {n}")
print(f"\nACTIVE_COLLECTIONS (download + BirdNET): {ACTIVE_COLLECTIONS}")


diff_genus:
  Fringilla coelebs
  Loxia curvirostra
  Chloris chloris
  Carduelis carduelis
  Linaria cannabina
  Pyrrhula pyrrhula
  Coccothraustes coccothraustes
  Spinus spinus
  Serinus serinus
  Carpodacus erythrinus

ACTIVE_COLLECTIONS (download + BirdNET): ['diff_genus']


# Download from Xeno-canto

In [ ]:
from building.download import download_and_process

for collection_name, species_names in collections.items():
    await download_and_process(species_names, 
        collection_name, 
        clips_per_species=CLIPS_PER_SPECIES, 
        max_recordings=MAX_RECORDINGS,
        auto_delete=True,)

Fetching available listings...
[Fringilla coelebs] available=910 processed=11 queued=899
[Loxia curvirostra] available=299 processed=0 queued=299
[Chloris chloris] available=166 processed=0 queued=166
[Carduelis carduelis] available=154 processed=0 queued=154
[Linaria cannabina] available=100 processed=0 queued=100
[Pyrrhula pyrrhula] available=66 processed=0 queued=66
[Coccothraustes coccothraustes] available=75 processed=0 queued=75
[Spinus spinus] available=76 processed=0 queued=76
[Serinus serinus] available=81 processed=0 queued=81
[Carpodacus erythrinus] available=249 processed=0 queued=249


# BirdNET validation + dataset assembly

In [ ]:
from building.dataset import build_dataset

MAX_PER_CLASS = 1750
dataset_paths = {}
for coll_name, species_names in collections_to_use.items():
    dataset_paths[coll_name] = build_dataset(coll_name, species_names, clips_per_species=CLIPS_PER_SPECIES, max_class_size_training=MAX_PER_CLASS)
    print(f"  {coll_name}: {dataset_paths[coll_name]}")

  diff_genus: /home/nathan/Documents/multi-chirp/datasets/diff_genus


# Sanity check

In [ ]:

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips")
        for species, count in sorted(species_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {species:20s} {count:5d}")


diff_genus
  training      9971 clips
    Myiarchus_tyrannulus  2089
    Pitangus_sulphuratus  1234
    Attila_spadiceus      1072
    Tyrannus_melancholicus   987
    Megarynchus_pitangua   953
    Myiodynastes_maculatus   845
    Myiozetetes_similis    652
    non_target             644
    Lathrotriccus_euleri   516
    Camptostoma_obsoletum   495
    Tolmomyias_sulphurescens   484
  validation    3073 clips
    Myiarchus_tyrannulus   636
    Pitangus_sulphuratus   432
    non_target             351
    Tyrannus_melancholicus   328
    Attila_spadiceus       260
    Megarynchus_pitangua   252
    Myiodynastes_maculatus   221
    Myiozetetes_similis    174
    Tolmomyias_sulphurescens   173
    Camptostoma_obsoletum   136
    Lathrotriccus_euleri   110
  testing       2906 clips
    Myiarchus_tyrannulus   618
    Pitangus_sulphuratus   368
    Tyrannus_melancholicus   352
    non_target             323
    Attila_spadiceus       270
    Myiodynastes_maculatus   221
    Megarynchus_p